In [2]:
# ===== BASELINE GREEDY — CONFIG (edit ONLY this cell) =====================

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "test/stable-ac-moves-w4"   # branch that contains experiments/
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push
MOUNT_DRIVE = True           # results -> Drive if True, else local ./results/

# --- experiment knobs ---------------------------------------------------
BUDGET = [1000, 5000, 10000, 25000, 50000]   # node budget per presentation; one jsonl per budget

cfg = {
    "DATASET": "data/ms640_solved.txt",
    "SUBSET": None,                 # None=all 640, list[int], or (start, end)
    "MAX_RELATOR_LENGTH": 24,       # per-relator cap (24 = ms640 layout)
    "CYCLIC_REDUCE": True,          # toggle cyclic reduction after each move

    # jsonl field toggles (each keeps BOTH the length + the relator pair)
    "use_min_relator": True,            # keeps min_relator_length + min_relator
    "use_max_relator": True,            # keeps max_relator_length + max_relator
    "use_max_relator_expanded": True,   # keeps max_relator_length_expanded + max_relator_expanded (longest state popped/expanded)
    "use_time": True,
    "use_path": True,
    "PATH_IN_SEPARATE_FILE": True,  # solved paths -> *_paths.jsonl

    "RESUME": True,                 # skip pres_ids already in the jsonl

    "PROGRESS_EVERY": 10,           # print a status line every N presentations

    # output
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "DRIVE_OUT_DIR": "/content/drive/MyDrive/acsolverx_results/greedy_baseline",
    "LOCAL_OUT_DIR": "results/greedy_baseline",

    # Weights & Biases
    "USE_WANDB": True,
    "AUTO_AUTHENTICATE_WANDB": True,  # True: authenticate once & reuse key while runtime is alive.
                                     # False: prompt for the API key on EVERY run of the SETUP cell (to switch/refresh a key).
    "WANDB_ENTITY": "avigyapaudel045-aisc",   # writable team entity (org-managed acct); None = account default
    "WANDB_PROJECT": "acsolver",
    "WANDB_MODE": "online",         # "offline" for flaky Colab -> wandb sync later
    "WANDB_GROUP": None,            # default: greedy_baseline_{date}
}

print("config loaded.")

config loaded.


In [ ]:
# ==================== SETUP (clone / pull / install / mount) ==============
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy wandb")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# run from repo root so "data/..." and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# ---- W&B authentication -------------------------------------------------
# AUTO_AUTHENTICATE_WANDB (set in CONFIG):
#   True  -> PROMPTLESS auth from a Colab Secret named WANDB_API_KEY (persists
#            across runtime restarts). Create it ONCE: Colab left sidebar ->
#            key icon (Secrets) -> add WANDB_API_KEY = <your key> -> toggle
#            'Notebook access' ON. Without the secret it falls back to a
#            one-time paste for this session (so it keeps asking until added).
#   False -> prompt for the API key EVERY run (use to switch/refresh a key).
if cfg["USE_WANDB"]:
    import re, wandb

    def _clean_key(k):
        # strip whitespace/newlines and stray surrounding quotes from a paste
        return (k or "").strip().strip('"').strip("'").strip()

    def _prompt_key():
        import getpass
        return _clean_key(getpass.getpass(
            "Paste your W&B API key (from wandb.ai/authorize), then Enter: "))

    _auto = cfg.get("AUTO_AUTHENTICATE_WANDB", True)
    if not _auto:
        os.environ["WANDB_API_KEY"] = _prompt_key()          # always ask for a fresh key
    elif not os.environ.get("WANDB_API_KEY"):
        _key = None
        _why = ""
        try:
            from google.colab import userdata
            _key = _clean_key(userdata.get("WANDB_API_KEY"))
        except Exception as _e:
            _why = type(_e).__name__   # SecretNotFoundError / NotebookAccessError
        if _key:
            os.environ["WANDB_API_KEY"] = _key
            print("W&B: using Colab Secret WANDB_API_KEY (promptless).")
        else:
            print("W&B: AUTO_AUTHENTICATE_WANDB=True but no usable Colab Secret "
                  f"WANDB_API_KEY{(' (' + _why + ')') if _why else ''}.")
            print("     -> Promptless auth: Colab left sidebar -> key icon (Secrets)")
            print("        -> Add  name=WANDB_API_KEY  value=<key from wandb.ai/authorize>")
            print("        -> toggle 'Notebook access' ON, then re-run this cell.")
            print("     Pasting once for THIS session instead:")
            os.environ["WANDB_API_KEY"] = _prompt_key()

    # catch a malformed paste BEFORE calling the server (common Colab getpass issue)
    _k = os.environ.get("WANDB_API_KEY", "")
    if not re.fullmatch(r"[A-Za-z0-9_]+", _k):
        print("W&B: that key has invalid characters (allowed: A-Z a-z 0-9 _).")
        print("     Re-copy it EXACTLY from https://wandb.ai/authorize "
              "-- no spaces, quotes, or a 'wandb login' prefix.")

    # verify; relogin=True overwrites any stale key cached in ~/.netrc
    try:
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True, verify=True)
        try:
            _default = wandb.Api().default_entity
        except Exception:
            _default = None
        _target = cfg["WANDB_ENTITY"] or _default
        print(f"W&B: authenticated ✓  runs -> {_target}/{cfg['WANDB_PROJECT']}")
    except Exception as e:
        print(f"W&B: authentication FAILED ✗ -- {e}")
        print("     tip: add a Colab Secret WANDB_API_KEY (promptless), or set "
              "AUTO_AUTHENTICATE_WANDB=False and re-run to paste a fresh key.")

In [4]:
# ==================== RUN =================================================
from experiments.run_baseline import run_dataset

for budget in BUDGET:
    run_dataset(cfg, node_budget=budget)   # resumable; one jsonl per budget

=== budget=1000 | 640 presentations | 0 already done, 640 to run | -> /content/drive/MyDrive/acsolverx_results/greedy_baseline/greedy_1000_640_mrl24_cyc_all_07_09_26.jsonl
    [1000] 10/640 | solved 10/10 (100.0%) | pres 9: ok nodes=4 | 4s elapsed, ETA 240s (2.6/s)
    [1000] 20/640 | solved 20/20 (100.0%) | pres 19: ok nodes=8 | 4s elapsed, ETA 119s (5.2/s)
    [1000] 30/640 | solved 30/30 (100.0%) | pres 29: ok nodes=7 | 4s elapsed, ETA 79s (7.8/s)
    [1000] 40/640 | solved 40/40 (100.0%) | pres 39: ok nodes=4 | 4s elapsed, ETA 58s (10.3/s)
    [1000] 50/640 | solved 50/50 (100.0%) | pres 49: ok nodes=11 | 4s elapsed, ETA 46s (12.7/s)
    [1000] 60/640 | solved 60/60 (100.0%) | pres 59: ok nodes=4 | 4s elapsed, ETA 38s (15.2/s)
    [1000] 70/640 | solved 70/70 (100.0%) | pres 69: ok nodes=8 | 4s elapsed, ETA 32s (17.6/s)
    [1000] 80/640 | solved 80/80 (100.0%) | pres 79: ok nodes=5 | 4s elapsed, ETA 28s (19.9/s)
    [1000] 90/640 | solved 90/90 (100.0%) | pres 89: ok nodes=9 | 4s 

SystemError: CPUDispatcher(<function get_neighbors_with_moves_nj at 0x7f366cda6020>) returned a result with an exception set